# Figure S1

Alternative-landscape strategy robustness across NK, Rough Mount Fuji, and stochastic block landscapes. Raw sweep products live in `raw_data/`, processed summaries live in `processed_data/`, and figures are saved to `figures/{pdf,png,eps}/`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pickle
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure

from slide.utils import (
    FIGURE_LABEL_SIZE,
    FIGURE_LEGEND_SIZE,
    FIGURE_TICK_SIZE,
    FIGURE_TITLE_SIZE,
    PANEL_LETTER_SIZE,
    get_figures_dir,
    get_processed_data_dir,
    get_raw_data_dir,
    load_pickle,
    save_pickle,
)

OVERWRITE_RAW_PKL: bool = False
OVERWRITE_PROCESSED_PKL: bool = False
PLOT_ONLY: bool = True
SAVE_FIGURES: bool = True
PANEL_DPI: int = 350
SAVE_TYPES: tuple[str, ...] = ("pdf", "png", "eps")

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
REPO_ROOT = Path.cwd()

LANDSCAPE_FAMILIES: tuple[str, ...] = ("NK", "RMF", "Block")
RUGGEDNESS_LABELS: tuple[str, ...] = ("Low", "Mid", "High")
STRATEGY_FILENAMES: dict[str, str] = {
    "NK": "landscape_comparsion_strategy_sweep_NK.pkl",
    "RMF": "landscape_comparsion_strategy_sweep_RMF.pkl",
    "Block": "landscape_comparsion_strategy_sweep_block.pkl",
}
DECAY_FILENAMES: dict[str, str] = {
    "NK": "alt_landscapes_decay_curve_sweep_NK.pkl",
    "RMF": "alt_landscapes_decay_curve_sweep_RMF.pkl",
    "Block": "alt_landscapes_decay_curve_sweep_blocks.pkl",
}
GENERATOR_SCRIPTS: dict[str, str] = {
    "landscape_comparsion_strategy_sweep_NK.pkl": "scripts/strategy_sweep_alt_landscapes_NK.py",
    "landscape_comparsion_strategy_sweep_RMF.pkl": "scripts/strategy_sweep_alt_landscapes_RMF.py",
    "landscape_comparsion_strategy_sweep_block.pkl": "scripts/strategy_sweep_alt_landscapes_block.py",
    "alt_landscapes_decay_curve_sweep_NK.pkl": "scripts/decay_curve_sweep_alt_landscapes_NK.py",
    "alt_landscapes_decay_curve_sweep_RMF.pkl": "scripts/decay_curve_sweep_alt_landscapes_RMF.py",
    "alt_landscapes_decay_curve_sweep_blocks.pkl": "scripts/decay_curve_sweep_alt_landscapes_block.py",
}
PROCESSED_PATH = PROCESSED_DATA_DIR / "figureS1_alt_landscapes_processed.pkl"

print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")
print(f"PLOT_ONLY={PLOT_ONLY}, OVERWRITE_RAW_PKL={OVERWRITE_RAW_PKL}, OVERWRITE_PROCESSED_PKL={OVERWRITE_PROCESSED_PKL}")


## Helpers

In [ ]:
def raw_path(filename: str) -> Path:
    """Return a Figure S1 raw-data path.

    Parameters:
    - filename: str
        Raw pickle filename.

    Returns:
    - Path
        Full path below ``RAW_DATA_DIR``.
    """
    return RAW_DATA_DIR / filename


def save_figure(fig: Figure, stem: str, *, bbox_inches: str = "tight") -> None:
    """Save a figure in every configured format.

    Parameters:
    - fig: Figure
        Matplotlib figure to save.
    - stem: str
        Output filename stem without extension.
    - bbox_inches: str
        Bounding-box mode passed to Matplotlib.

    Returns:
    - None
        Files are written below ``FIGURES_DIR``.
    """
    if not SAVE_FIGURES:
        return
    for suffix in SAVE_TYPES:
        destination = FIGURES_DIR / suffix
        destination.mkdir(parents=True, exist_ok=True)
        fig.savefig(destination / f"{stem}.{suffix}", dpi=PANEL_DPI, bbox_inches=bbox_inches)


def add_panel_letter(ax: Axes, letter: str) -> None:
    """Add a manuscript panel letter to an axis.

    Parameters:
    - ax: Axes
        Axis receiving the label.
    - letter: str
        Panel letter.

    Returns:
    - None
        The axis is modified in place.
    """
    ax.text(-0.12, 1.08, letter, transform=ax.transAxes, fontsize=PANEL_LETTER_SIZE, fontweight="bold", va="top", ha="left")


def load_bare_pickle(path: Path) -> object:
    """Load a bare pickle object from an explicit path.

    Parameters:
    - path: Path
        Pickle path to load.

    Returns:
    - object
        Deserialized object.
    """
    with path.open("rb") as handle:
        return pickle.load(handle)


## Raw Products

In [ ]:
def run_raw_generator(filename: str) -> None:
    """Run the existing S1 generator for one raw product.

    Parameters:
    - filename: str
        Raw filename whose mapped generator should run.

    Returns:
    - None
        The generator writes the raw product into ``RAW_DATA_DIR``.
    """
    script = GENERATOR_SCRIPTS[filename]
    env = os.environ.copy()
    env["SLIDE_DATA_DIR"] = str(RAW_DATA_DIR)
    command = [sys.executable, script]
    print("Running:", " ".join(command), f"with SLIDE_DATA_DIR={RAW_DATA_DIR}")
    subprocess.run(command, cwd=REPO_ROOT, env=env, check=True)


def ensure_raw_products() -> dict[str, dict[str, np.ndarray]]:
    """Load or generate all Figure S1 raw products.

    Parameters:
    - None

    Returns:
    - dict[str, dict[str, np.ndarray]]
        Raw strategy and decay arrays keyed by landscape family.
    """
    raw_payloads: dict[str, dict[str, np.ndarray]] = {"strategy": {}, "decay": {}}
    required_filenames = tuple(STRATEGY_FILENAMES.values()) + tuple(DECAY_FILENAMES.values())
    missing = [filename for filename in required_filenames if not raw_path(filename).exists()]
    if PLOT_ONLY:
        print("PLOT_ONLY=True: skipping raw generation/loading.")
        if missing:
            print("Missing raw files, but processed data may still be loaded:")
            for filename in missing:
                print(f"  {raw_path(filename)}")
        return raw_payloads

    for filename in required_filenames:
        path = raw_path(filename)
        if OVERWRITE_RAW_PKL or not path.exists():
            run_raw_generator(filename)
        if not path.exists():
            raise FileNotFoundError(f"Generator did not create required raw file: {path}")

    for family in LANDSCAPE_FAMILIES:
        raw_payloads["strategy"][family] = np.asarray(load_bare_pickle(raw_path(STRATEGY_FILENAMES[family])), dtype=float)
        raw_payloads["decay"][family] = np.asarray(load_bare_pickle(raw_path(DECAY_FILENAMES[family])), dtype=float)
    return raw_payloads


raw_payloads = ensure_raw_products()


## Processing

In [ ]:
def process_figure_s1_payload(raw_payloads: dict[str, dict[str, np.ndarray]]) -> dict[str, object]:
    """Reduce raw S1 sweeps into plotting summaries.

    Parameters:
    - raw_payloads: dict[str, dict[str, np.ndarray]]
        Raw strategy and decay arrays keyed by landscape family.

    Returns:
    - dict[str, object]
        Processed heatmaps, optima, normalized decay curves, and provenance.
    """
    processed: dict[str, dict[str, object]] = {}
    expected_strategy_shape = (3, 7, 7, 300)
    expected_decay_shape = (3, 25, 1, 25)
    for family in LANDSCAPE_FAMILIES:
        strategy = np.asarray(raw_payloads["strategy"][family], dtype=float)
        decay = np.asarray(raw_payloads["decay"][family], dtype=float)
        if strategy.shape != expected_strategy_shape:
            raise ValueError(f"{family} strategy sweep has shape {strategy.shape}; expected {expected_strategy_shape}.")
        if decay.shape != expected_decay_shape:
            raise ValueError(f"{family} decay sweep has shape {decay.shape}; expected {expected_decay_shape}.")
        if not np.all(np.isfinite(strategy)) or not np.all(np.isfinite(decay)):
            raise ValueError(f"{family} raw S1 arrays contain non-finite values.")

        strategy_means = strategy.mean(axis=3)
        optima = np.empty((len(RUGGEDNESS_LABELS), 2), dtype=int)
        for ruggedness_index in range(len(RUGGEDNESS_LABELS)):
            y_max, x_max = np.unravel_index(np.argmax(strategy_means[ruggedness_index]), strategy_means[ruggedness_index].shape)
            optima[ruggedness_index] = (x_max, y_max)

        decay_means = decay.mean(axis=(1, 2))
        decay_normalized = decay_means / decay_means[:, [0]]
        processed[family] = {
            "strategy_means": strategy_means,
            "optima_xy": optima,
            "decay_means": decay_means,
            "decay_normalized": decay_normalized,
            "strategy_raw_path": str(raw_path(STRATEGY_FILENAMES[family])),
            "decay_raw_path": str(raw_path(DECAY_FILENAMES[family])),
        }

    return {
        "data": processed,
        "params": {
            "landscape_families": LANDSCAPE_FAMILIES,
            "ruggedness_labels": RUGGEDNESS_LABELS,
            "strategy_shape": expected_strategy_shape,
            "decay_shape": expected_decay_shape,
            "strategy_filenames": STRATEGY_FILENAMES,
            "decay_filenames": DECAY_FILENAMES,
            "generator_scripts": GENERATOR_SCRIPTS,
        },
        "metadata": {
            "paper_reference": "Figure S1",
            "description": "Alternative-landscape strategy heatmaps and decay-curve summaries.",
            "source_script": "scripts/plot_figure_s1.py",
            "raw_products": "Bare pickle arrays loaded from raw_data.",
        },
    }


if PROCESSED_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    figure_s1_payload = load_pickle(PROCESSED_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(f"PLOT_ONLY=True requires processed payload {PROCESSED_PATH}")
else:
    figure_s1_payload = process_figure_s1_payload(raw_payloads)
    save_pickle(figure_s1_payload, PROCESSED_PATH)

for family in LANDSCAPE_FAMILIES:
    panel = figure_s1_payload["data"][family]
    assert np.asarray(panel["strategy_means"]).shape == (3, 7, 7)
    assert np.asarray(panel["optima_xy"]).shape == (3, 2)
    assert np.asarray(panel["decay_normalized"]).shape == (3, 25)
print(f"Validated processed Figure S1 payload: {PROCESSED_PATH}")


## Plot

In [ ]:
def plot_strategy_panel(ax: Axes, heatmap: np.ndarray, optima: np.ndarray, ruggedness_index: int, row_index: int, column_index: int) -> None:
    """Plot one strategy heatmap panel.

    Parameters:
    - ax: Axes
        Axis receiving the heatmap.
    - heatmap: np.ndarray
        Two-dimensional strategy-performance matrix.
    - optima: np.ndarray
        ``(ruggedness, xy)`` optimum coordinates.
    - ruggedness_index: int
        Index of the ruggedness level being plotted.
    - row_index: int
        Landscape-family row index.
    - column_index: int
        Strategy-column index.

    Returns:
    - None
        Artists are added directly to ``ax``.
    """
    ax.imshow(heatmap)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([0, 6])
    ax.set_yticks([0, 6])
    ax.tick_params(axis="both", which="major", labelsize=FIGURE_TICK_SIZE)
    if row_index == len(LANDSCAPE_FAMILIES) - 1:
        ax.set_xticklabels([0.0, 0.19])
        ax.set_xlabel("Base chance", fontsize=FIGURE_LABEL_SIZE)
    else:
        ax.set_xticklabels([])
    if column_index == 0:
        ax.set_yticklabels([24, 1])
        ax.set_ylabel("No. sub populations", fontsize=FIGURE_LABEL_SIZE)
    else:
        ax.set_yticklabels([])
    x_max, y_max = optima[ruggedness_index]
    ax.plot(x_max, y_max, "o", color="white", markersize=5, markeredgecolor="black", markeredgewidth=0.8, zorder=3)
    if ruggedness_index > 0:
        xs = optima[: ruggedness_index + 1, 0]
        ys = optima[: ruggedness_index + 1, 1]
        ax.plot(xs, ys, "-", color="white", linewidth=1.5, alpha=0.9, zorder=2)


def plot_decay_panel(ax: Axes, decay_normalized: np.ndarray, row_index: int) -> None:
    """Plot normalized decay curves for one landscape family.

    Parameters:
    - ax: Axes
        Axis receiving the decay curves.
    - decay_normalized: np.ndarray
        Normalized decay curves with shape ``(ruggedness, generation)``.
    - row_index: int
        Landscape-family row index.

    Returns:
    - None
        Artists are added directly to ``ax``.
    """
    for ruggedness_index, curve in enumerate(decay_normalized):
        ax.plot(curve, label=RUGGEDNESS_LABELS[ruggedness_index])
    ax.tick_params(axis="both", which="major", labelsize=FIGURE_TICK_SIZE)
    ax.legend(fontsize=FIGURE_LEGEND_SIZE)
    if row_index == len(LANDSCAPE_FAMILIES) - 1:
        ax.set_xlabel("Generations", fontsize=FIGURE_LABEL_SIZE)
    else:
        ax.set_xticklabels([])
    ax.set_ylabel("Fitness (a.u.)", fontsize=FIGURE_LABEL_SIZE)


def plot_figure_s1(payload: dict[str, object]) -> Figure:
    """Render the complete Figure S1 panel grid.

    Parameters:
    - payload: dict[str, object]
        Processed Figure S1 payload.

    Returns:
    - Figure
        Rendered Matplotlib figure.
    """
    fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(16, 10), constrained_layout=True, dpi=PANEL_DPI)
    column_labels = (*RUGGEDNESS_LABELS, "Decay curves")
    for row_index, family in enumerate(LANDSCAPE_FAMILIES):
        family_data = payload["data"][family]
        strategy_means = np.asarray(family_data["strategy_means"], dtype=float)
        optima = np.asarray(family_data["optima_xy"], dtype=int)
        for column_index in range(3):
            plot_strategy_panel(axes[row_index, column_index], strategy_means[column_index], optima, column_index, row_index, column_index)
        plot_decay_panel(axes[row_index, 3], np.asarray(family_data["decay_normalized"], dtype=float), row_index)
        axes[row_index, 0].annotate(family, xy=(-0.25, 0.5), xycoords="axes fraction", fontsize=15, ha="right", va="center", rotation=90, fontweight="bold")
    for column_index, label in enumerate(column_labels):
        axes[0, column_index].set_title(label, fontsize=FIGURE_TITLE_SIZE, fontweight="bold", pad=12)
    fig.text(0.39, 1.01, "Strategy space for landscapes of increasing ruggedness", ha="center", va="bottom", fontsize=18, fontweight="bold")
    letters = "ABCDEFGHIJKL"
    for index, ax in enumerate(axes.flat):
        add_panel_letter(ax, letters[index])
    return fig


fig = plot_figure_s1(figure_s1_payload)
if SAVE_FIGURES:
    save_figure(fig, "figure_S1")
plt.show()
